# Drone platforms

A survey of complete aircraft — who builds drones relevant to indoor
autonomous capture, what each actually does, and what each lets you change.
The [flight compute notebook](08_flight_compute_landscape.ipynb) covers boards
and modules; this one covers things that fly out of the box, from $200
Shenzhen cinewhoops to six-figure mining scanners. So far our platform
research has centered on [ModalAI](07_voxl2_mini.ipynb); this notebook is the
deliberate look around.

The organizing axis is **what the vendor lets you change**, because for a
research-then-product effort that matters more than any spec:

1. **Turnkey, closed** — finished capability, no code access. Useful as
   competitive benchmarks and as proof of what is tractable.
2. **Closed autopilot, open payload** — DJI-class platforms: you may bolt on
   compute and sensors, but the flight brain is a black box.
3. **Open research platforms** — PX4/ArduPilot-based aircraft sold to be
   reprogrammed; the flight stack, companion computer, and sensors are yours.
4. **Bare airframes** — the FPV (first-person view) industry's parts
   ecosystem; everything is yours, including the labor.

Core take: **autonomy is bought locked or built open — nobody sells it
unlocked.** Every platform that ships real indoor autonomy ships it closed;
every open platform ships with the autonomy left as an exercise. That split
is the market's shape, and it is why the earlier notebooks keep concluding
that the differentiating work is the [goal-selection
layer](13_navigation_exploration.ipynb), not the vehicle.

Prices are August 2026 snapshots where a source is named, and omitted where
verification failed; the
[volatility caveats](08_flight_compute_landscape.ipynb#caveats-on-the-figures-above)
from the compute notebook apply here in full.

## Turnkey confined-space and indoor inspection platforms

The category closest to our problem: aircraft sold specifically to fly where
GPS is not, and bring back data.

| Platform | Origin | What it is | Autonomy | Price signal |
|---|---|---|---|---|
| [Flyability Elios 3](https://www.flyability.com/elios-3) | CH | Cage-protected quad, 4K + thermal + [ouster lidar], 12 min, IP44 | Assisted teleop; collision-*tolerant* by cage, not by planning | ~$25k ([reseller](https://robotomated.com/explore/drone/flyability-elios-3)) |
| [Emesent Hovermap ST-X](https://www.emesent.com/emesent-product/hovermap-series/) | AU | Lidar SLAM *payload* (300 m range, ~1 M pts/s) on a DJI M300/M350 or handheld | True bounded-volume autonomous exploration ("draw a cube, it scans it") | payload alone runs tens of thousands of dollars; carrier drone extra |
| [Exyn ExynAero](https://www.exyn.com/) | US | Lidar quad for mines/stockpiles | Self-declared "Level 4" volume exploration | enterprise-quote only |
| [Skydio X10](https://www.skydio.com/x10/technical-specs) | US | Six nav cameras, 50/64 MP + FLIR Boson+ thermal, NightSense zero-light flight, onboard 2D/3D mapping | Best-in-class obstacle avoidance + 3D Scan coverage of an operator-defined volume | ~$15.7k ([tracker](https://robotomated.com/explore/drone/skydio-x10d)) |
| [Skydio R10](https://www.skydio.com/r10) | US | Purpose-built *indoor* variant, ~770 g, integrated lighting, two-way audio; GA targeted H1 2026 | Skydio autonomy indoors; public-safety framing | unannounced |
| [BRINC Lemur 2](https://brincdrones.com/lemur-2/) | US | Public-safety indoor quad: lidar + 4K + FLIR, glass-breaker option, 20+ min | Real-time lidar 2D floor plans / 3D map ("Autonomy Engine"); pilot in command | quote-only |
| [Cleo Robotics Dronut X1](https://cleorobotics.com/) | US | Bi-rotor *ducted donut*, 425 g, truly collision-safe, 12 min | Assisted teleop + position hold | quote-only |
| [Leica BLK2FLY](https://shop.leica-geosystems.com/blk2fly) | CH | Flying laser scanner (survey-grade lidar + 5 cameras) | Autonomous *scan coverage* of building exteriors; indoor GPS-denied supported | ~$50k class (unverified) |

[ouster lidar]: https://ouster.com/

Reading the table against
[the autonomy scorecard](08_flight_compute_landscape.ipynb#the-autonomy-scorecard):
only Emesent and Exyn *explore* — both lidar-first, mining-bred, with
geometry (not imagery) as the deliverable. Skydio and Leica *cover* an
operator-drawn volume with excellent imagery but no exploration. Flyability,
BRINC, and Cleo are *flown* — their value is surviving the environment, and
their maps are situational awareness, not reconstruction products. The
pattern from the compute notebook holds after this wider sweep: exploration
players produce point clouds, imagery players need a human to point them.

Two design lessons worth stealing regardless: the cage/duct (Elios, Dronut)
converts obstacle contact from a crash into a bump, which changes the safety
calculus of flying near furniture entirely; and every serious indoor
platform ships its own lighting, conceding that ambient indoor light is
insufficient for their cameras — while still leaving the harsh
view-dependent shading problem ([capture-quality gap #3](08_flight_compute_landscape.ipynb#where-the-gap-is))
unsolved.

## Warehouse inventory drones: autonomy at scale, no 3D deliverable

A separate turnkey niche that matters as an existence proof.
[Verity](https://www.verity.net/) (CH),
[Corvus Robotics](https://www.corvus-robotics.com/) (US), and
[Gather AI](https://www.gather.ai/) (US) sell fully autonomous indoor
drones that fly inventory scans in working warehouses — lights on or off, no
GPS, no beacons, no pilot, launching and landing from charging nests on a
schedule. Corvus shipped a freezer-rated variant in early 2026 (deployed at
Kroger, −20 °F operation); Verity documents customers halving inventory
costs.

What makes them relevant despite the different mission: they are the only
vendors anywhere shipping **unattended indoor autonomy as a product** —
takeoff to landing with nobody watching. It is tractable because the
environment cooperates: warehouse aisles are straight, tall, static,
rack-textured corridors, and the deliverable is barcode reads at known rack
positions, not geometry. Nobody has carried this level of autonomy into
furnished, cluttered, room-scale space — that, plus a reconstruction-grade
deliverable, is exactly the open square on the board.

## Camera drones: closed autopilot, open payload

The volume market. Superb hardware, sophisticated *assistive* autonomy, and
a locked flight stack.

**DJI** owns the category (its 2026 line: Mini 5 Pro / Neo 2 / Avata 2
consumer; Mavic 4 Pro prosumer; [Matrice 4](https://enterprise.dji.com/matrice-4-series)
and [Matrice 400](https://enterprise.dji.com/matrice-400) enterprise). The
enterprise flagships are formidable sensor platforms — the Matrice 400 flies
59 min, lifts 6 kg, and carries rotating lidar + mmWave (millimeter-wave)
radar + low-light omnidirectional vision for obstacle sensing. But across
the whole line the obstacle system is *avoidance*, not mapping; "terrain
follow" and "waypoint" autonomy assume outdoor missions; and developer
access is the [Payload SDK](https://developer.dji.com/payload-sdk/) —
mount your computer as a *payload*, receive telemetry and video, inject
guidance-level commands — while the flight controller, state estimator, and
obstacle logic stay sealed (the old Onboard SDK that gave Matrice-series
low-level control is discontinued). Emesent's choice to ship Hovermap as a
DJI payload shows both what PSDK enables and what it costs: their SLAM must
*fight* the sealed DJI estimator rather than replace it.

**[Autel Robotics](https://www.autelrobotics.com/)** (CN, Shenzhen) is the
DJI-shaped alternative; the EVO Max 4T V2 explicitly markets GPS-denied
indoor flight (stereo vision + mmWave radar, "Autonomy Engine" path
planning, anti-jamming). Same closure model as DJI.

**[Parrot ANAFI Ai](https://www.parrot.com/en/drones/anafi-ai)** (FR) is the
odd one out and worth knowing: a 900 g stereo-vision drone whose
[Air SDK](https://developer.parrot.com/) runs *your code on the drone
itself* with access to cameras, IMU, and the autopilot's planning layer,
plus the free Sphinx simulator (Gazebo + Unreal). The closest thing to an
open enterprise camera drone — from a vendor whose consumer exit and small
market share make longevity the standing question.

**Skydio** left the consumer market in 2023 to sell autonomy to enterprise
and defense (previous section) — evidence that best-in-class obstacle
avoidance alone could not win a consumer camera-drone fight against DJI's
price/quality curve.

For our purposes this category contributes sensors-per-dollar benchmarks and
one strategic fact: **no camera-drone vendor sells what our project needs to
exist** — if they did, the [gap analysis](08_flight_compute_landscape.ipynb#where-the-gap-is)
would already be closed.

## Open research platforms: sold to be reprogrammed

Aircraft whose selling point is that the software is yours. The practical
question for each is what you get beyond the parts: calibration, working
VIO/mapping out of the box, documentation, community.

- **[ModalAI Starling 2 / Starling 2 Max](https://www.modalai.com/products/starling-2)**
  (US) — the [VOXL 2](07_voxl2_mini.ipynb) sold as a flying dev kit:
  ~280–500 g, factory-calibrated cameras, qVIO and voxblox mapping working
  on first boot, PX4 on-SoC. The only open platform that ships *working
  indoor autonomy plumbing* rather than parts. Sentinel is the larger
  stereo-camera sibling. Weaknesses per notebook 07: closed VIO algorithm,
  aging Ubuntu/ROS base, VOXL 3 transition risk.
- **[Holybro X500 v2 PX4 dev kit](https://holybro.com/products/px4-development-kit-x500-v2)**
  (CN) — the reference Pixhawk-class build: carbon frame, Pixhawk 6C/6X,
  assembles in ~30 min, no soldering; add your own companion computer and
  cameras. The [PX4 Vision v1.5](https://holybro.com/products/px4-vision-dev-kit-v1-5)
  variant adds an UP Core computer and a Structure Core depth camera with
  PX4's obstacle-avoidance sample stack. Cheap, standard, entirely open —
  and entirely un-integrated: VIO, mapping, and calibration are your
  problem.
- **[AMOVLAB](https://www.amovlab.top/) P450 / P600** (CN, Chengdu) — PX4 +
  ROS research quads around their open-source
  [Prometheus](https://github.com/amov-lab/Prometheus) autonomy framework
  (planning, SLAM, swarm, SpireCV vision), sold with onboard computers and
  depth/lidar options. A Dronecode member; the P-series is widespread in
  Chinese university labs — the closest Chinese analog to a Starling,
  built from open parts.
- **[Bitcraze Crazyflie 2.1 brushless](https://www.bitcraze.io/)** (SE) —
  the 30–40 g open-everything nano platform: schematics, firmware, Python
  API, lighthouse/UWB (ultra-wideband) positioning, swarm tooling. Too
  small to carry capture sensors; unbeatable for cheap algorithm work and
  swarm research at desk scale.
- **[Agilicious](https://github.com/uzh-rpg/agilicious)** (UZH) — the
  open-source, open-hardware agile-flight reference (Science Robotics
  2022): 5-inch racing frame, Jetson TX2-class compute, supports
  neural-network controllers. Built for the racing/agility research line,
  not for mapping payloads; the CTU-Prague [MRS UAV system](https://github.com/ctu-mrs/mrs_uav_system)
  plays the same role for multi-UAV field research on larger frames.

Graveyard, as a calibration on platform longevity: Intel Aero (dead 2019),
Qualcomm Flight (dead, [resurrected only as boards](08_flight_compute_landscape.ipynb)),
DJI Matrice 100 + Onboard SDK (the last time DJI sold programmable flight,
discontinued), 3DR Solo (dead 2016, PX4's corporate parent exited hardware).
Open *platforms* die fast; the open *stacks* (PX4, ArduPilot, ROS) survive
their hardware hosts — an argument for keeping value in the stack, echoed in
[the ROS notes](11_ros.ipynb).

## The FPV industry: airframes as a parts bin

One tier below "platform": the first-person-view hobby industry — almost
entirely Chinese (iFlight, GEPRC, BetaFPV, Flywoo, Diatone, SpeedyBee,
Foxeer; plus US-designed frames built on the same supply chain) — sells
exactly the airframe class an indoor capture drone needs, at hobby prices.

The relevant subspecies is the **cinewhoop**: a 2.5–3.5-inch quad with full
prop ducts, tuned for smooth slow flight in tight indoor spaces, built to
bump into things and carry a camera — GEPRC's Cinelog series is the
benchmark, BetaFPV's Pavo line gets brushless HD capture under 100 g, and
build guides for the class are commodity content. This is the same
airworthiness concept as the Elios cage and Dronut duct, at 1/50th the
price, minus every gram of autonomy.

Why it matters beyond cheap frames: the FPV stack (Betaflight flight
controllers, 20×20/30×30 mm mounting standards, O4-class digital video)
is *convertible*. The same frames fly ArduPilot/PX4 on FPV-format flight
controllers (Chinese vendors like MicoAir ship ArduPilot-targeted boards),
and ModalAI's own Starling is essentially a cinewhoop-class frame around a
VOXL. The realistic prototype path for a house-scale capture drone is a
ducted 3–3.5-inch frame + [Pixhawk-class FMU or VOXL](08_flight_compute_landscape.ipynb#the-split-stack-pixhawk-fc-separate-computer)
+ companion — commodity muscle and skeleton under a research brain. The
FPV parts bin is also the fallback that keeps any single-vendor risk
(ModalAI's roadmap, DJI's lockdown, covered-list churn) from being
existential: motors, ESCs, frames, and video links are multi-sourced
commodities.

## The Chinese ecosystem

Worth treating as a system rather than a vendor list, because it is where
most of the world's drone hardware actually comes from.

**Scale.** Shenzhen alone hosts ~1,900 companies in the "low-altitude
economy"; Chinese consumer drones hold ~70% of the global market and
industrial drones ~50%; Guangdong produced ~6.9 M civil UAVs in 2024, and
the 15th Five-Year Plan (Oct 2025) made the sector a national strategic
priority. DJI sits on the deepest component ecosystem in the industry —
motors, gimbals, batteries, radios, cameras — which is why nobody matches
its price/quality curve.

**The layers, from our buyer's perspective:**

- **Finished autonomy-adjacent drones**: DJI and Autel (previous section) —
  closed.
- **The open-hardware autopilot industry is Chinese.** Holybro and CUAV
  build most of the world's
  [Pixhawk-standard](10_pixhawk_ecosystem.ipynb#the-standards-themselves)
  autopilots; SIYI (radios, gimbals, now autopilots), Radiolink, and
  MicoAir fill the rest of the catalog; the entire FPV component industry
  (previous section) rounds it out. "US-made" alternatives exist at 3–10×
  the price for compliance reasons, not capability — the
  [compute notebook](08_flight_compute_landscape.ipynb) prices that
  premium.
- **Research platforms**: AMOVLAB's Prometheus P-series (previous section);
  AMOV RobotLab also resells ModalAI hardware into China (a price source
  [notebook 09](09_modalai_modules.ipynb) already uses). Chinese university
  labs (HKUST, ZJU-FAST — the groups behind
  [Fast-Planner, EGO-Planner, FUEL, RACER](13_navigation_exploration.ipynb#systems-worth-knowing-by-name))
  publish the open planning software the rest of the world's drones run;
  hardware and algorithms both flow out of the same ecosystem.
- **Capture hardware**: [XGRIDS](https://xgrids.com/) (Shenzhen) sells the
  splat-native handheld scanners (next section) — currently the most
  product-shaped 3DGS capture devices anywhere.

**The regulatory squeeze, both directions.** In the US: DJI (and Autel)
were added to the FCC Covered List in December 2025 after the NDAA's
(National Defense Authorization Act) audit deadline lapsed, and the FCC
categorically extended coverage to foreign-made drones and *critical
components* — blocking new equipment authorizations (existing aircraft keep
flying; no retroactive grounding). In the other direction, China has applied
export controls to drone components with military applicability since 2023.
Practical consequences for a project like ours: US-market product plans
cannot assume Chinese airframes or radios remain importable; everywhere
else, the Chinese supply chain remains the default and the cheapest path;
and either way the *software* — the part we intend to own — is unaffected.

## Handheld capture: the non-flying competition

The devices that currently *win* the indoor-capture deliverable, included
because they define the quality bar and the price ceiling a drone product
must beat:

- **[XGRIDS Lixel L2 Pro](https://xgrids.com/lixell2)** — lidar (16/32
  channel) + 48 MP panoramic cameras + IMU, multi-SLAM, and native 3D
  Gaussian Splatting output through their LCC pipeline (walk the building,
  get an explorable splat scene for Unity/UE5/web). ~€5.6–6.3k at EU
  resellers; the sub-kg PortalCam pushes the same idea cheaper. The most
  direct commercial expression of the deliverable our
  [reconstruction notes](02_3d_photo_reconstruction.ipynb#neural-rendering-nerf-and-3d-gaussian-splatting)
  target.
- **[Leica BLK2GO](https://shop.leica-geosystems.com/blk2go)** — the
  survey-grade handheld SLAM scanner (~$50k class); BLK2FLY is its flying
  sibling.
- **[NavVis VLX](https://www.navvis.com/)** — wearable scanner for
  AEC-grade (architecture/engineering/construction) building capture.
- **[Matterport Pro3](https://matterport.com/)** — tripod lidar camera; the
  incumbent of real-estate capture, human-repositioned per scan point.

The strategic read: handhelds solve capture *quality* by attaching the
sensor to a human who solves navigation, coverage, and lighting judgment
for free. They are cheaper, safer, and better-lit than any drone today —
in reachable spaces. A capture drone's case rests on what a walking human
cannot do: unreachable volumes, hazardous interiors, elevated viewpoints,
scheduled unattended re-capture, and eventually cost-per-scan at fleet
scale. Any pitch that ignores the $6k walking alternative is dishonest
about its competition.

## Fit to the indoor capture project

What this wider sweep changes and confirms:

- **The gap survives the broader look.** Adding BRINC, Cleo, Autel, Parrot,
  the warehouse players, and the Chinese research vendors to the picture
  surfaces no one combining autonomous exploration of furnished interiors
  with reconstruction-grade imagery. The closest approaches come from
  opposite sides: Emesent (real exploration, lidar deliverable) and XGRIDS
  (real splat deliverable, human navigation).
- **Development platform: unchanged.** Starling 2 (working autonomy
  plumbing, one vendor) versus Holybro-class split stack plus companion
  (everything open, everything manual) remains the real choice, with
  AMOVLAB's P-series now a named third option of the second kind — worth a
  closer look if procurement from China is acceptable, since it ships the
  ROS/planning integration Holybro leaves out.
- **Airframe endgame: ducted, cinewhoop-class.** Elios, Dronut, Starling,
  and the entire cinewhoop industry independently converged on protected
  props for indoor flight; a product airframe should start there, built
  from the commodity FPV supply chain.
- **Two design patterns to adopt**: contact tolerance as a safety layer
  (cage/duct changes the cost of the planner being wrong), and onboard
  lighting as a capture subsystem to engineer deliberately rather than
  bolt on — no vendor has solved capture-grade illumination, so it is
  open differentiation surface.
- **Benchmarks to hold ourselves against**: Skydio X10 for
  obstacle-avoidance quality, Hovermap for bounded-volume exploration UX
  ("draw a cube"), Verity/Corvus for unattended operation, XGRIDS LCC for
  the deliverable. Each defines the bar on one axis; the product thesis is
  clearing all four at house scale.
- **Supply-chain posture**: keep the differentiating value in software and
  the vehicle multi-sourced. The FCC covered-list expansion makes *every*
  foreign airframe a US-market risk, which strengthens the case for a
  stack that ports across [MAVLink/ROS 2](11_ros.ipynb) vehicles rather
  than marrying any one of them.

## References

Turnkey and inspection:

- [Flyability Elios 3](https://www.flyability.com/elios-3);
  [reseller spec/price snapshot](https://robotomated.com/explore/drone/flyability-elios-3)
- [Emesent Hovermap series](https://www.emesent.com/emesent-product/hovermap-series/);
  [ST-X exploration-mode field report](https://candrone.com/blogs/news/exploring-the-unreachable-emesent-s-hovermap-st-x-at-candrone-demo-day-2025)
- [Exyn Technologies](https://www.exyn.com/)
- [Skydio X10 specs](https://www.skydio.com/x10/technical-specs);
  [Skydio R10](https://www.skydio.com/r10)
- [BRINC Lemur 2](https://brincdrones.com/lemur-2/);
  [launch coverage](https://uavcoach.com/brinc-lemur-2/)
- [Cleo Robotics Dronut](https://cleorobotics.com/)
- [Leica BLK2FLY](https://shop.leica-geosystems.com/blk2fly)
- [Verity](https://www.verity.net/), [Corvus Robotics](https://www.corvus-robotics.com/)
  ([cold-chain launch](https://www.roboticstomorrow.com/news/2026/02/09/corvus-robotics-launches-dedicated-cold-chain-drones-for-autonomous-inventory-in-sub-zero-warehouses/26115/)),
  [Gather AI](https://www.gather.ai/)

Camera drones and SDKs:

- [DJI Matrice 400](https://enterprise.dji.com/matrice-400),
  [Matrice 4 series](https://enterprise.dji.com/matrice-4-series),
  [Payload SDK](https://developer.dji.com/payload-sdk/),
  [Onboard SDK (discontinued)](https://developer.dji.com/onboard-sdk/)
- [Autel EVO Max 4T V2](https://www.autelpilot.com/products/autel-robotics-evo-max-4t-v2)
- [Parrot ANAFI Ai SDK program](https://www.parrot.com/us/drones/anafi-ai/technical-documentation/sdk),
  [developer docs](https://developer.parrot.com/)

Open platforms:

- [ModalAI Starling 2](https://www.modalai.com/products/starling-2)
- [Holybro X500 v2 PX4 dev kit](https://holybro.com/products/px4-development-kit-x500-v2),
  [PX4 Vision v1.5](https://holybro.com/products/px4-vision-dev-kit-v1-5)
- [AMOVLAB P600](https://www.amovlab.top/products/p600-research-uav-development-platform),
  [Prometheus](https://github.com/amov-lab/Prometheus),
  [Dronecode membership](https://dronecode.org/amovlab-joins-dronecode-foundation-to-join-effort-in-open-standards-adoption/)
- [Bitcraze Crazyflie](https://www.bitcraze.io/)
- [Agilicious](https://github.com/uzh-rpg/agilicious)
  ([paper](https://www.science.org/doi/10.1126/scirobotics.abl6259));
  [MRS UAV system](https://github.com/ctu-mrs/mrs_uav_system)
- [ROS aerial-robotics hardware landscape](https://ros-aerial.github.io/aerial_robotic_landscape/hardware/)

FPV and Chinese ecosystem:

- [Cinewhoop buyer's guide (Oscar Liang)](https://oscarliang.com/cinewhoop/)
- [Shenzhen low-altitude economy coverage](https://news.cgtn.com/news/2026-05-22/Shenzhen-s-low-altitude-economy-soars-as-city-leads-global-drone-race-1Nm7IY3OUbm/p.html);
  [Chinese drone market overview](https://droneii.com/drone-companies-in-the-chinese-drone-market)
- [DJI ban / FCC covered list status](https://uavcoach.com/dji-ban/)

Handheld capture:

- [XGRIDS Lixel L2 Pro](https://xgrids.com/lixell2);
  [EU reseller pricing](https://epotronic.com/eng/manufacturers/xgrids/)
- [Leica BLK2GO](https://shop.leica-geosystems.com/blk2go),
  [NavVis](https://www.navvis.com/), [Matterport](https://matterport.com/)

Related notes: [flight compute landscape](08_flight_compute_landscape.ipynb)
for boards and the autonomy scorecard, [VOXL 2 Mini](07_voxl2_mini.ipynb) for
the ModalAI deep dive, [navigation and exploration](13_navigation_exploration.ipynb)
for the software this hardware would run, and
[3D photo reconstruction](02_3d_photo_reconstruction.ipynb) for the
deliverable.